# Voxel Head Model

This notebook introduces `VoxelHeadModel`, an alternative to
`TwoSurfaceHeadModel` that represents the brain as a reduced set of
**voxels** rather than a triangulated cortex surface.  Voxel-based
reconstruction keeps depth information in the reconstructed image — a
useful property for high-density montages that reach the cortical and
subcortical volume.  See `40_image_reconstruction.ipynb` for the full
surface-based pipeline; this notebook focuses on what changes when the
brain is voxelised.

The pipeline mirrors the surface case:

1. Load a voxel head model from a standard atlas.
2. Snap optodes to the scalp; reduce voxels to those covered by the probe.
3. Run Monte-Carlo photon transport (or load precomputed fluence).
4. **Reduce voxels by fluence** — runs *between*
   MCX and `compute_sensitivity` so the resulting sensitivity matrix is
   small.
5. Compute the sensitivity matrix `Adot` directly on the kept voxels.
6. Optionally apply a final `reduce_voxels_to_sensitivity` for cleanup.
7. Reconstruct via `ImageRecon`.

In [ ]:
# This cell sets up the environment when executed in Google Colab.
try:
    import google.colab
    get_ipython().system('curl -s https://raw.githubusercontent.com/ibs-lab/cedalion/dev/scripts/colab_setup.py -o colab_setup.py')
    get_ipython().run_line_magic('run', 'colab_setup.py')
except ImportError:
    pass

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import xarray as xr

import cedalion
import cedalion.data
import cedalion.dataclasses as cdc
import cedalion.dot as dot
import cedalion.dot.forward_model as fw
import cedalion.nirs
from cedalion import units

xr.set_options(display_expand_data=False);

## 1. Load a voxel head model

`get_standard_headmodel(model, kind="voxel")` returns a `VoxelHeadModel`
built from the same Colin27/ICBM152 segmentation masks that back the
surface variant.  The brain is meshed as voxels rather than as a
triangulated cortex; the scalp surface is reused from the atlas.

In [ ]:
head = dot.get_standard_headmodel("colin27", kind="voxel")
print(head)

Compared to the surface variant, the brain attribute is now a
`cdc.Voxels` cloud rather than a mesh.  The sparse mapping
`voxel_to_vertex_brain` projects the full segmentation grid onto these
kept voxels — the columns are voxels here, not mesh vertices.  Code
that prefers a clearer name can use the alias
`voxel_to_voxel_brain`:

In [ ]:
print("brain voxels:", head.brain.nvertices)
print("voxel_to_vertex_brain shape:", head.voxel_to_vertex_brain.shape)
print("alias property identity:", head.voxel_to_voxel_brain is head.voxel_to_vertex_brain)

## 2. Probe-based reduction

Most montages cover only part of the head.  Brain voxels that no optode
can plausibly see can be dropped before any photon transport simulation.

In [ ]:
rec = cedalion.data.get_fingertappingDOT()

# snap the recording's optode positions to the scalp surface
geo3d_snapped = head.align_and_snap_to_scalp(rec.geo3d)

# drop voxels far from any optode
head_ijk = head.reduce_voxels_to_probe(geo3d_snapped, max_dist=4 * units.cm)
print("after reduce_voxels_to_probe:", head_ijk.brain.nvertices, "voxels")

## 3. Use precomputed fluence

The full Monte-Carlo step (`ForwardModel.compute_fluence_mcx`) requires
an NVIDIA GPU and is omitted from the notebook output.  We use a
precomputed fluence HDF5 file instead.  Note the fluence file lives on
the *full* segmentation voxel grid — independent of which head model
(surface or voxel) consumes it.

In [ ]:
fluence_fname = cedalion.data.get_precomputed_fluence("fingertappingDOT", "colin27")
print("fluence file:", fluence_fname)

## 4. Fluence reducer

`reduce_voxels_by_fluence` reads the fluence file, computes per-voxel
$\sum_\text{optodes} |\text{fluence}|$ (max over wavelengths) and drops
voxels below `rel_threshold * max`. Runs *before* `compute_sensitivity`, so the
resulting `Adot` is small from the start.

In [ ]:
head_ijk = head_ijk.reduce_voxels_by_fluence(fluence_fname, rel_threshold=1e-3)
print("after reduce_voxels_by_fluence:", head_ijk.brain.nvertices, "voxels")

## 5. Compute sensitivity

`ForwardModel.compute_sensitivity` works exactly the same on a voxel
head model — the duck-typed attribute names match.  The resulting
`Adot` has dims `(channel, vertex, wavelength)` where `vertex` indexes
the kept voxels.

In [ ]:
measurement_list = rec._measurement_lists["amp"]

fwm = fw.ForwardModel(head_ijk, geo3d_snapped, measurement_list)

with TemporaryDirectory() as tmp_dir:
    sens_path = Path(tmp_dir) / "sensitivity.nc"
    fwm.compute_sensitivity(fluence_fname, sens_path)
    Adot = cedalion.io.forward_model.load_Adot(sens_path)

print(Adot.dims, Adot.shape)
print("is_brain True count:", int(Adot.is_brain.sum()))

## 6. Optional final cleanup

After `compute_sensitivity`, voxels with negligible total sensitivity
can be removed via an absolute threshold.

In [ ]:
head_ijk_final = head_ijk.reduce_voxels_to_sensitivity(
    Adot, sensitivity_threshold=1e-4
)
print("after reduce_voxels_to_sensitivity:", head_ijk_final.brain.nvertices, "voxels")

## 7. Image reconstruction

`ImageRecon` accepts `Adot` for either head model variant.  Below we
build a Tikhonov-only solver and reconstruct a single time-point of
synthetic optical density to confirm the output dimensions land on the
voxel `vertex` axis.

In [ ]:
solver = dot.ImageRecon(
    Adot=Adot,
    recon_mode="conc",
    **dot.REG_TIKHONOV_ONLY,
)

# tiny synthetic OD: zero perturbation everywhere
n_channels = Adot.sizes["channel"]
n_wavelengths = Adot.sizes["wavelength"]
od = xr.DataArray(
    np.zeros((n_channels, n_wavelengths, 1), dtype=np.float32),
    dims=("channel", "wavelength", "time"),
    coords={
        "channel": Adot.channel,
        "wavelength": Adot.wavelength,
        "time": [0.0],
    },
    attrs={"units": ""},
).pint.quantify()
od.time.attrs["units"] = "s"

image = solver.reconstruct(od)
print(image.dims, image.shape)

## What's next

- See `40_image_reconstruction.ipynb` for the full surface-based
  pipeline with realistic stimulus epochs and 3D visualisation.
- Replace the synthetic OD above with a block-averaged response from
  the recording to produce a voxel HbO/HbR map.
- The `voxel_to_voxel_brain` alias gives semantic-friendly access to
  the projection matrix; for downstream computations the duck-typed
  `voxel_to_vertex_brain` is what `ForwardModel` and `ImageRecon`
  consume.